In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



In [3]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',None)



In [4]:
df = pd.read_csv('../data/complaints.csv', encoding='latin-1')

In [5]:
print("=========Shape=======")
print(df.shape)
print("=========First 5 Rows========")
print(df.head())

=========Shape=======
(2023066, 11)
=========First 5 Rows========
   Unnamed: 0         product_5  \
0         234  Credit Reporting   
1         240   Debt Collection   
2         257  Credit Reporting   
3         271  Credit Reporting   
4         276             Loans   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [6]:
print(df.dtypes)
print("\n")
print(df.columns.tolist())

Unnamed: 0          int64
product_5             str
narrative             str
Product               str
Date received         str
Sub-product           str
Issue                 str
Sub-issue             str
Company               str
State                 str
Timely response?      str
dtype: object


['Unnamed: 0', 'product_5', 'narrative', 'Product', 'Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Timely response?']


In [7]:
missing=df.isnull().sum()
missing_percentage=(missing/len(df))*100
print(missing_percentage)

Unnamed: 0           0.000000
product_5            0.000000
narrative            0.000000
Product              0.000000
Date received        0.000000
Sub-product          2.580539
Issue                0.000000
Sub-issue           11.396514
Company              0.000000
State                0.363013
Timely response?     0.000000
dtype: float64


In [8]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0])

             Missing Count  Missing %
Sub-issue           230559  11.396514
Sub-product          52206   2.580539
State                 7344   0.363013


In [9]:
print(df['product_5'].value_counts())

product_5
Credit Reporting              1205275
Debt Collection                266842
Loans                          228599
Credit Card Services           163710
Bank Accounts and Services     158640
Name: count, dtype: int64


In [10]:
print(f"Unique products : {df['product_5'].nunique()}")

Unique products : 5


In [11]:
df = df.rename(columns={'product_5': 'Complaint_Category'})
df = df.rename(columns={'narrative': 'Narrative'})

df['Timely_response'] = df['Timely response?'].map({
    'Yes': 1,
    'No': 0
})

df = df.rename(columns={'Date received': 'Date Received'})

df['Date Received'] = pd.to_datetime(df['Date Received'])
             

In [12]:
print(df.columns.tolist())

['Unnamed: 0', 'Complaint_Category', 'Narrative', 'Product', 'Date Received', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Timely response?', 'Timely_response']


In [13]:
print(df.head())

   Unnamed: 0 Complaint_Category  \
0         234   Credit Reporting   
1         240    Debt Collection   
2         257   Credit Reporting   
3         271   Credit Reporting   
4         276              Loans   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [14]:
df=df.drop(columns=['Timely response?'])
print(df.columns.tolist())

['Unnamed: 0', 'Complaint_Category', 'Narrative', 'Product', 'Date Received', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Timely_response']


In [15]:
samples = []
for category in df['Complaint_Category'].unique():
    sample = df[df['Complaint_Category'] == category].sample(n=10000, random_state=42)
    samples.append(sample)

df_sampled = pd.concat(samples).reset_index(drop=True)

print(df_sampled['Complaint_Category'].value_counts())
print(f"\nSampled dataset shape: {df_sampled.shape}")

Complaint_Category
Credit Reporting              10000
Debt Collection               10000
Loans                         10000
Bank Accounts and Services    10000
Credit Card Services          10000
Name: count, dtype: int64

Sampled dataset shape: (50000, 11)


In [16]:
import re

In [17]:
def assign_priority(text):
    if not isinstance(text,str):
        return 'Low'

    text_lower=text.lower()

    high_keywords = [
    'fraud', 'unauthorized', 'identity theft', 'legal action',
    'lawsuit', 'attorney', 'sue', 'stolen', 'scam', 'harassment',
    'threatening', 'criminal', 'police', 'emergency', 'foreclosure',

    # security / account compromise
    'account hacked', 'hacked', 'compromised', 'breach', 'data breach',
    'security breach', 'account takeover', 'unauthorized access',
    'suspicious login', 'unknown device', 'password changed',
    'email compromised', 'phishing', 'phished', 'malware', 'virus',

    # financial fraud / charge abuse
    'credit card fraud', 'card stolen', 'bank fraud', 'payment fraud',
    'unauthorized charge', 'unauthorized transaction', 'stolen funds',
    'money missing', 'funds removed', 'chargeback fraud',

    # legal / escalation language
    'legal escalation', 'report to authorities', 'report police',
    'consumer protection', 'regulator', 'lawsuit pending',
    'court action', 'legal complaint',

    # urgency / severe impact
    'urgent', 'immediately', 'asap', 'right away', 'critical',
    'severely impacted', 'can’t access account', 'locked out',
    'completely blocked', 'service down', 'total outage',

    # sensitive harm / threats
    'threat', 'extortion', 'blackmail', 'abuse', 'assault',
    'danger', 'unsafe', 'life threatening'
]
    

    medium_keywords = [
    'incorrect', 'error', 'dispute', 'inaccurate', 'wrong',
    'billing', 'overcharged', 'refund', 'not resolved', 'ignored',
    'still waiting', 'no response', 'unresolved',

    # billing & payments (non-fraud)
    'invoice issue', 'billing error', 'payment failed', 'payment issue',
    'double charged', 'extra charge', 'subscription issue',
    'renewal issue', 'pricing question', 'charge question',
    'refund request', 'partial refund', 'credit request',

    # service quality issues
    'slow service', 'poor service', 'not working properly',
    'glitch', 'bug', 'technical issue', 'system error',
    'downtime', 'intermittent issue', 'performance issue',

    # access issues (non-security)
    'login issue', 'cannot login', 'can’t log in', 'password reset',
    'reset password', 'locked out', 'access issue', 'verification issue',

    # support delays / dissatisfaction
    'delayed response', 'no update', 'waiting for reply',
    'follow up', 'escalation request', 'still pending',
    'not satisfied', 'bad experience', 'frustrated',

    # account / subscription management
    'cancel subscription', 'cancellation issue', 'upgrade issue',
    'downgrade issue', 'plan change', 'account settings'
]

    if any(word in text_lower for word in high_keywords):
        return 'High'
    elif any(word in text_lower for word in medium_keywords):
        return 'Medium'
    else:
        return 'Low'


df_sampled['Priority'] = df_sampled['Narrative'].apply(assign_priority)
print(df_sampled['Priority'].value_counts())

    

Priority
High      25273
Low       16039
Medium     8688
Name: count, dtype: int64


In [19]:
import ssl
import certifi

ssl._create_default_https_context = ssl.create_default_context
import nltk
import re
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ''
    
 
    text = text.lower()
    

    text = re.sub(r'x+', '', text)
    

    text = re.sub(r'[^a-z\s]', '', text)
    

    text = re.sub(r'\s+', ' ', text).strip()
    

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    return ' '.join(tokens)

df_sampled['Cleaned_Narrative'] = df_sampled['Narrative'].apply(clean_text)


print("ORIGINAL:")
print(df_sampled['Narrative'].iloc[0][:300])
print("\nCLEANED:")
print(df_sampled['Cleaned_Narrative'].iloc[0][:300])

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bardwellmazambara/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


ORIGINAL:
By the provisions of the Fair Credit Reporting Act, I am submitting this complaint to request that the listed account which I believe appears to be inaccurate and incomplete be immediately investigated and corrected on my credit file. 
XXXX XXXX

CLEANED:
provisions fair credit reporting act submitting complaint request listed account believe appears inaccurate incomplete immediately investigated corrected credit file


In [21]:
df_sampled=df_sampled.drop(columns=['Narrative'])

In [23]:
print(df_sampled.columns.tolist())

['Unnamed: 0', 'Complaint_Category', 'Product', 'Date Received', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Timely_response', 'Priority', 'Cleaned_Narrative']
